# DPS with Scanline Masking — Diffusion vs Flow Matching

Mirrors the *Active perception for focused transmit steering* notebook from `zea`, but uses the two locally trained checkpoints in this repository.

**Task:** Simulate acquiring only a sparse set of focused scan lines (vertical columns in the polar domain) and use DPS to reconstruct the full image from each model.

**Pipeline per model:**
1. Draw an unconditional sample as the reference frame.
2. Apply a scanline mask (evenly-spaced vertical columns).
3. Run `posterior_sample` (DPS) to reconstruct.
4. Compare: reference | masked measurement | DPS reconstruction(s) | posterior variance.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from zea import init_device
init_device(verbose=True)

In [ ]:
import numpy as np
import keras
import keras.ops as ops
import matplotlib.pyplot as plt

from zea.models.diffusion import DiffusionModel  # noqa: F401
from zea.models.flow_matching import FlowMatchingModel  # noqa: F401
from zea.ops.ultrasound import scan_convert
from zea.visualize import set_mpl_style

set_mpl_style()

## Configuration

In [ ]:
# ── Sampling ──────────────────────────────────────────────────────────────────
N_STEPS          = 50    # DPS reverse-diffusion steps
N_POSTERIOR      = 4     # posterior samples per measurement
OMEGA            = 10.0  # DPS step-size weight (matches agent_example.ipynb)
SEED_DPS         = 42    # seed for DPS

# ── Scanline mask ─────────────────────────────────────────────────────────────
# Fraction of columns to acquire (25% matches the notebook's 14/56 lines)
SCANLINE_FRACTION = 0.05

# ── Scan-convert geometry (must match training data) ─────────────────────────
RHO_RANGE   = (0.0, 0.15)
THETA_RANGE = (-np.pi / 6, np.pi / 6)

# ── Checkpoints ───────────────────────────────────────────────────────────────
MODELS = {
    "Diffusion":     "checkpoints/diffusion.keras",
    "Flow Matching": "checkpoints/flow_matching.keras",
}

CUSTOM_OBJECTS = {
    "DiffusionModel":     DiffusionModel,
    "FlowMatchingModel":  FlowMatchingModel,
}

## Helpers

In [ ]:
def make_scanline_mask(H, W, C, fraction=0.25, seed=0):
    """Return a float32 scanline mask of shape (1, H, W, C).

    Each selected column (scan line in the polar domain) is fully revealed
    (value 1); all other columns are masked out (value 0).
    Lines are evenly spaced across the width.

    Args:
        H, W, C: Image height, width, channels.
        fraction: Fraction of columns to keep.
        seed: RNG seed for reproducibility.

    Returns:
        mask: float32 array of shape (1, H, W, C).
        selected_cols: indices of the selected columns.
    """
    n_lines = max(1, int(W * fraction))
    # Evenly space lines across the full width
    selected_cols = np.round(np.linspace(0, W - 1, n_lines)).astype(int)
    mask = np.zeros((H, W, C), dtype=np.float32)
    mask[:, selected_cols, :] = 1.0
    return mask[np.newaxis], selected_cols  # (1, H, W, C)


def to_sc(img_hwc):
    """Scan-convert a single (H, W, C) image → (H', W') numpy array."""
    sc, _ = scan_convert(
        np.squeeze(img_hwc),
        rho_range=RHO_RANGE,
        theta_range=THETA_RANGE,
        fill_value=0.0,
    )
    return sc


def imshow(ax, img_hwc, title="", vmin=0.0, vmax=1.0, cmap="gray"):
    """Display scan-converted image on ax."""
    ax.imshow(to_sc(img_hwc), cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=8, pad=3)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

## Load a validation frame

We load one real frame from the **EchoNet-LVH** validation set.
This single frame is used as the shared reference for **both** models,
so the DPS reconstructions are directly comparable.

In [ ]:
import sys
sys.path.insert(0, "/flowmatching")
from dataloader import build_dataset

VAL_PATH   = "/data/USBMD_datasets/EchoNet-LVH/val"
IMG_SIZE   = (256, 256)   # must match model input_shape
FRAME_IDX  = 0            # which frame to use as reference

val_ds = build_dataset(VAL_PATH, image_size=IMG_SIZE, batch_size=32, shuffle=False)
batch = next(iter(val_ds))         # (32, 256, 256, 1)  values in [0, 1]
ref_np = batch[FRAME_IDX].numpy()  # (256, 256, 1)

print(f"Reference frame: shape={ref_np.shape}, min={ref_np.min():.3f}, max={ref_np.max():.3f}")

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
imshow(ax, ref_np, title="Reference (val frame)")
plt.tight_layout()
plt.show()

## Run DPS for both models

Both models reconstruct from the **same** real validation frame.
Columns: **Reference** | **Measurement** (sparse scanlines) | **Posterior samples** | **Variance**

In [ ]:
results = {}  # keyed by model name
H, W, C = ref_np.shape  # 256, 256, 1

# Build the shared scanline mask once (same for both models)
mask, selected_cols = make_scanline_mask(H, W, C, fraction=SCANLINE_FRACTION)
measurement = ref_np[np.newaxis] * mask  # (1, H, W, C)
print(f"Mask: {len(selected_cols)} / {W} columns ({100 * SCANLINE_FRACTION:.0f}%)")

for model_name, ckpt_path in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  {model_name}  —  {ckpt_path}")
    print(f"{'=' * 60}")

    model = keras.models.load_model(ckpt_path, custom_objects=CUSTOM_OBJECTS)

    print(f"  Running DPS ({N_STEPS} steps, {N_POSTERIOR} samples, ω={OMEGA}) …")
    posteriors = model.posterior_sample(
        measurements=measurement,   # (1, H, W, C)
        mask=mask,                  # (1, H, W, C)
        n_samples=N_POSTERIOR,
        n_steps=N_STEPS,
        omega=OMEGA,
        seed=keras.random.SeedGenerator(SEED_DPS),
        verbose=True,
    )
    # posteriors: (1, N_POSTERIOR, H, W, C)
    posteriors_np = np.array(posteriors)[0]  # (N_POSTERIOR, H, W, C)

    # Posterior variance in scan-convert space
    sc_stack = np.stack([to_sc(posteriors_np[i]) for i in range(N_POSTERIOR)], axis=0)
    variance_sc = np.var(sc_stack, axis=0)

    results[model_name] = {
        "ref": ref_np,
        "measurement": measurement[0],
        "posteriors": posteriors_np,
        "variance_sc": variance_sc,
    }

print("\nDone.")

In [ ]:
N_ROWS = len(MODELS)
N_COLS = 2 + N_POSTERIOR + 1  # ref + measurement + N_POSTERIOR + variance

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(N_COLS * 2.2, N_ROWS * 2.6),
    gridspec_kw={"wspace": 0.05, "hspace": 0.15},
)
if N_ROWS == 1:
    axes = axes[np.newaxis, :]

col_titles = (
    ["Reference", f"Measurement\n({100 * SCANLINE_FRACTION:.0f}% lines)"]
    + [f"DPS sample {i + 1}" for i in range(N_POSTERIOR)]
    + ["Posterior\nvariance"]
)

for row, (model_name, res) in enumerate(results.items()):
    # row label
    axes[row, 0].set_ylabel(model_name, fontsize=10, rotation=90, labelpad=6, va="center")

    imshow(axes[row, 0], res["ref"],         title=col_titles[0] if row == 0 else "")
    imshow(axes[row, 1], res["measurement"], title=col_titles[1] if row == 0 else "")

    for i in range(N_POSTERIOR):
        imshow(axes[row, 2 + i], res["posteriors"][i], title=col_titles[2 + i] if row == 0 else "")

    # variance panel (no scan-convert needed — already computed)
    ax_var = axes[row, -1]
    ax_var.imshow(res["variance_sc"], cmap="inferno", vmin=0)
    if row == 0:
        ax_var.set_title(col_titles[-1], fontsize=8, pad=3)
    ax_var.set_xticks([])
    ax_var.set_yticks([])
    for spine in ax_var.spines.values():
        spine.set_visible(False)

fig.suptitle(
    f"DPS — Scanline Inpainting  (n_steps={N_STEPS}, ω={OMEGA}, {100 * SCANLINE_FRACTION:.0f}% lines)",
    fontsize=12,
    y=1.01,
)

out_path = "checkpoints/dps_scanlines.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

---
## n_steps sweep — reconstruction MSE over 1 000 validation samples

For each model and each step count we:
1. Take a batch of real EchoNet-LVH validation frames.
2. Apply the same 5 % scanline mask.
3. Run DPS (`posterior_sample`, 1 draw per frame for speed).
4. Compute pixel-wise MSE between the reconstruction and the original frame.

The sweep reveals how reconstruction quality improves with more steps, and whether diffusion or flow matching converges faster.

In [ ]:
# ── Config for the sweep ──────────────────────────────────────────────────────
N_EVAL_SAMPLES   = 250
EVAL_BATCH_SIZE  = 10
N_STEPS_LIST     = [5, 10, 20, 50]
N_POSTERIOR_EVAL = 4      # single draw per frame (speed)
EVAL_SEED        = 7

# ── Load N_EVAL_SAMPLES frames from the val set ───────────────────────────────
print(f"Loading {N_EVAL_SAMPLES} validation frames …")
val_ds_eval = build_dataset(
    VAL_PATH, image_size=IMG_SIZE, batch_size=EVAL_BATCH_SIZE, shuffle=False
)

eval_frames_list = []
collected = 0
for batch in val_ds_eval:
    eval_frames_list.append(batch.numpy())
    collected += len(batch)
    if collected >= N_EVAL_SAMPLES:
        break

eval_frames = np.concatenate(eval_frames_list, axis=0)[:N_EVAL_SAMPLES]  # (N, H, W, 1)
print(f"Loaded {len(eval_frames)} frames — shape {eval_frames.shape}, "
      f"range [{eval_frames.min():.2f}, {eval_frames.max():.2f}]")

In [ ]:
from tqdm.auto import tqdm

# ── n_steps sweep ─────────────────────────────────────────────────────────────
sweep_results = {}   # {model_name: {n_steps: mean_mse}}

_H, _W, _C = eval_frames.shape[1:]
mask_1, _ = make_scanline_mask(_H, _W, _C, fraction=SCANLINE_FRACTION)

n_batches = int(np.ceil(N_EVAL_SAMPLES / EVAL_BATCH_SIZE))
total_iters = len(MODELS) * len(N_STEPS_LIST) * n_batches

pbar = tqdm(total=total_iters, unit="batch")

for model_name, ckpt_path in MODELS.items():
    model = keras.models.load_model(ckpt_path, custom_objects=CUSTOM_OBJECTS)
    sweep_results[model_name] = {}

    for n_steps in N_STEPS_LIST:
        mse_accum = []

        for b in range(n_batches):
            pbar.set_description(f"{model_name}  n_steps={n_steps:>3d}  batch {b+1}/{n_batches}")

            batch_ref = eval_frames[b * EVAL_BATCH_SIZE : (b + 1) * EVAL_BATCH_SIZE]
            bs = len(batch_ref)
            mask_b = np.tile(mask_1, (bs, 1, 1, 1))
            meas_b = batch_ref * mask_b

            posteriors_b = model.posterior_sample(
                measurements=meas_b,
                mask=mask_b,
                n_samples=N_POSTERIOR_EVAL,
                n_steps=n_steps,
                omega=OMEGA,
                verbose=False,
            )
            # shape: (bs, N_POSTERIOR_EVAL, H, W, C) → take first draw
            recon = np.array(posteriors_b)[:, 0, ...]
            mse = np.mean((recon - batch_ref) ** 2, axis=(1, 2, 3))
            mse_accum.extend(mse.tolist())
            pbar.update(1)

        mean_mse = float(np.mean(mse_accum))
        sweep_results[model_name][n_steps] = mean_mse
        pbar.write(f"  {model_name}  n_steps={n_steps:>4d}  →  mean MSE = {mean_mse:.5f}")

pbar.close()
print("\nSweep complete.")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))

for model_name, step_dict in sweep_results.items():
    steps = sorted(step_dict)
    mses  = [step_dict[s] for s in steps]
    ax.plot(steps, mses, marker="o", linewidth=2, label=model_name)

ax.set_xlabel("Number of DPS steps")
ax.set_ylabel("Mean MSE (reconstruction vs. reference)")
ax.set_title(
    f"DPS reconstruction quality vs. n_steps\n"
    f"({N_EVAL_SAMPLES} val frames, {100 * SCANLINE_FRACTION:.0f}% scanlines, ω={OMEGA})"
)
ax.set_xscale("log")
ax.set_xticks(N_STEPS_LIST)
ax.set_xticklabels(N_STEPS_LIST)
ax.legend()
fig.tight_layout()

sweep_path = "checkpoints/dps_nsteps_sweep.png"
fig.savefig(sweep_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {sweep_path}")